# hnc on Colab

Run the full Mapillary -> TRIBE v2 -> GeoParquet 2.0 pipeline on a Colab Pro **L4 24GB** runtime.

**Before you start**

1. **Runtime > Change runtime type > L4 GPU**.
2. Open the **Secrets** tab (key icon, left sidebar) and add:
   - `MAPILLARY_ACCESS_TOKEN` (required, from https://www.mapillary.com/dashboard/developers)
   - `HF_TOKEN` (optional, from https://huggingface.co/settings/tokens with read access). Without it, Hugging Face downloads run unauthenticated and may hit lower rate limits, but TRIBE v2 weights are public so the pipeline still works.
3. Run all cells top to bottom.

Repo, https://github.com/walkthru-earth/hnc

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf hnc
!git clone --depth 1 https://github.com/walkthru-earth/hnc.git
%cd /content/hnc

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version

In [ ]:
from google.colab import userdata
import os

def _get_optional_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

mapillary_token = _get_optional_secret('MAPILLARY_ACCESS_TOKEN')
if not mapillary_token:
    raise RuntimeError(
        "MAPILLARY_ACCESS_TOKEN is required. Add it under the Secrets tab "
        "(key icon in the left sidebar) and rerun this cell."
    )
os.environ['MAPILLARY_ACCESS_TOKEN'] = mapillary_token

hf_token = _get_optional_secret('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token

# Colab pre-sets MPLBACKEND='module://matplotlib_inline.backend_inline'
# which leaks into uv subprocesses and breaks mne -> matplotlib import
# during TRIBE v2 load. Force a headless backend.
os.environ['MPLBACKEND'] = 'Agg'

with open('.env', 'w') as f:
    f.write(f"MAPILLARY_ACCESS_TOKEN={os.environ['MAPILLARY_ACCESS_TOKEN']}\n")
    if hf_token:
        f.write(f"HF_TOKEN={hf_token}\n")
    f.write("MPLBACKEND=Agg\n")

print(
    f"secrets loaded (HF_TOKEN={'set' if hf_token else 'missing, using anonymous HF downloads'}), "
    ".env written, MPLBACKEND forced to Agg"
)

In [ ]:
!uv sync --extra tribe 2>&1 | tail -20

In [ ]:
# Override torch with Blackwell-compatible cu128 wheels in the project venv.
# Without -p .venv/bin/python, `uv pip install` lands in the system /usr env
# and `uv run` keeps using the original torch from .venv (sm_50..sm_90 only).
# Colab Pro can hand you an RTX PRO 6000 Blackwell (sm_120). cu128 wheels
# include sm_120 on top of all older capabilities, so this is safe on Hopper,
# Ada, and Ampere too.
!uv pip install -p /content/hnc/.venv/bin/python --reinstall \
  --index-url https://download.pytorch.org/whl/cu128 \
  torch torchvision 2>&1 | tail -10
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'arch_list', torch.cuda.get_arch_list())"

In [ ]:
!uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
!MPLBACKEND=Agg uv run hnc-run \
  --bbox-name borough \
  --max-images 50 \
  --start-captured-at 2020-01-01 \
  --device cuda \
  --out /content/hnc_borough.parquet

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
df = con.execute("SELECT image_id, captured_at, lon, lat, compass_angle, len(brain_activity) AS n_vertices FROM read_parquet('/content/hnc_borough.parquet') LIMIT 10").fetchdf()
df

In [ ]:
from google.colab import files
files.download('/content/hnc_borough.parquet')